# Machine Learning system for the Telegram chat bot

This notebook analyzes participant hashtag responses using sentence embeddings to measure semantic similarity, identify thematic structure, and build an interpretable chatbot response system based on anchor themes.

## Analysis Roadmap

This notebook proceeds in four steps:

1. Restructure the raw hashtag data into long format
2. Compute sentence embeddings for participant responses
3. Measure semantic similarity across scenarios and over time
4. Map responses onto interpretable theme anchors for chatbot response selection

### 1. Data Preparation

The raw dataset contains participant hashtag responses spread across multiple columns. This data was collected from a prior experiment of Hunter Priniski.
   
To make analysis easier, the data is reshaped into long format so that each row corresponds to one participant response.

This section:
- loads the dataset
- renames the participant ID column
- identifies hashtag columns
- reshapes the data
- cleans hashtag text
- extracts scenario IDs

In [36]:
import pandas as pd

df = pd.read_csv("8_narrative_pilot.csv")
df.head()

,Duration (in seconds),What is your Prolific ID?\n\nPlease note that this response should auto-fill with the correct ID,perspective_genetic testing,tweet,hashtag_1,hashtag_2,hashtag_3,hashtag_4,hashtag_5,hashtag_6,...,hashtag_4.7,hashtag_5.7,hashtag_6.7,hashtag_7.7,hashtag_8.7,hashtag_9.7,hashtag_10.7,Please enter your age.,Please indicate your gender.,Did you give this task an honest effort? Your answer will not impact your payment. We need to know for data analysis purposes.
0,685,5c56964991c0ad0001cfa5ae,2.0,Influenza is very dangerous and can spread qui...,#safety,#genes,#pandemic,#influenza,#scary,#health,...,#communityissues,#spiritual,#homeless,#casanueva,#historicalpreservation,#history,#politics,37,1,2
1,769,55ecd6ff748092000daa9f5f,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,38,1,2
2,860,6980e62c2e406af9a4bd1fb0,3.0,I'm not entirely sure what to think about this...,#genetics,#genetictesting,#storedgeneticinformation,#Reliant,#controversy,#consent,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,1,2
3,1059,6543d2b9d5cef1c67a0aa176,5.0,This is absolutely crazy that anyone would thi...,#crazy talk,#no rights,mandatory my ass,#no way,#sheep,#bumb,...,#noooooo,#noooooo,#no,#no[oooo,#nooooo,#nooooo,#noooooo,52,2,2
4,884,697d04d89f1cbdf87eeab1f1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,#applestohouses,#orchardsoflove,#growwithus,#letsgetgrowing,#casaorchard,#casagrow,#homecasaforall,32,2,2


In [35]:
#Rename messy ID column
id_col = df.columns[1]

df = df.rename(columns={id_col: "participant_id"})

In [5]:
# Identify all hashtag columns (columns starting with 'hashtag')
hashtag_cols = [col for col in df.columns if "hashtag" in col]

hashtag_cols[:10]

['hashtag_1',
 'hashtag_2',
 'hashtag_3',
 'hashtag_4',
 'hashtag_5',
 'hashtag_6',
 'hashtag_7',
 'hashtag_8',
 'hashtag_9',
 'hashtag_10']

In [6]:
#Expand to long format
long_df = df.melt(
    id_vars=["participant_id"],
    value_vars=hashtag_cols,
    var_name="hashtag_slot",
    value_name="hashtag"
)

long_df.head()

,participant_id,hashtag_slot,hashtag
0,5c56964991c0ad0001cfa5ae,hashtag_1,#safety
1,55ecd6ff748092000daa9f5f,hashtag_1,NaN
2,6980e62c2e406af9a4bd1fb0,hashtag_1,#genetics
3,6543d2b9d5cef1c67a0aa176,hashtag_1,#crazy talk
4,697d04d89f1cbdf87eeab1f1,hashtag_1,NaN


In [7]:
import re
#Clean Hashtags lowercase and remove non-alphanumeric characters
def clean_hashtag(tag):
    if pd.isna(tag):
        return None
    tag = str(tag).lower().replace("#", "").strip()
    tag = re.sub(r"[^a-z0-9]+", "", tag)
    return tag if tag != "" else None

long_df["hashtag"] = long_df["hashtag"].apply(clean_hashtag)

In [8]:
# Remove rows with missing participant IDs or hashtags, then reset the index
long_df = long_df.dropna(subset=["participant_id", "hashtag"])
long_df = long_df.reset_index(drop=True)

long_df.head()

,participant_id,hashtag_slot,hashtag
0,5c56964991c0ad0001cfa5ae,hashtag_1,safety
1,6980e62c2e406af9a4bd1fb0,hashtag_1,genetics
2,6543d2b9d5cef1c67a0aa176,hashtag_1,crazytalk
3,5dd2f94a238a913096c52317,hashtag_1,flu
4,5fa4f94e535f3b1aa628e0ab,hashtag_1,influenza


In [9]:
# Extract the prompt (scenario) number from hashtag_slot (each number represents which question the response belongs to)
long_df["scenario"] = long_df["hashtag_slot"].str.extract(r"\.(\d+)").fillna(0)
long_df["scenario"] = long_df["scenario"].astype(int)

In [39]:
print(long_df.shape)
long_df.head(100)

(15917, 6)


,participant_id,hashtag_slot,hashtag,scenario,embedding,round
0,5c56964991c0ad0001cfa5ae,hashtag_1,safety,0,"[0.00057354796, 0.07829996, -0.057217155, -0.0...",1
1,6980e62c2e406af9a4bd1fb0,hashtag_1,genetics,0,"[-0.03428222, 0.04356844, -0.052452214, 0.0873...",1
2,6543d2b9d5cef1c67a0aa176,hashtag_1,crazytalk,0,"[-0.00644761, -0.030833134, 0.011760357, -0.02...",1
3,5dd2f94a238a913096c52317,hashtag_1,flu,0,"[-0.018272426, 0.025064262, 0.0023866957, 0.05...",1
4,5fa4f94e535f3b1aa628e0ab,hashtag_1,influenza,0,"[-0.046492066, 0.024335867, -0.039698552, 0.06...",1
...,...,...,...,...,...,...
95,5c101f812a1a620001f04007,hashtag_1,autonomyisimportant,0,"[0.05575491, 0.009817158, -0.0030618648, -0.02...",1
96,56e6a66af6ed900006a5867c,hashtag_1,ethicalhealthcare,0,"[-0.053439658, 0.088995, -0.04479258, -0.04529...",1
97,615993153e17441c2e8d6c92,hashtag_1,autonomyinhealthcare,0,"[-0.025972921, 0.013839391, -0.007377153, -0.0...",1
98,67e07f19ee9c325f1bcaa026,hashtag_1,dnaanalysis,0,"[-0.113952205, 0.060711216, -0.03905395, -0.04...",1


### 2. Embedding Hashtag Responses

To compare short hashtag responses semantically, this notebook uses the `all-MiniLM-L6-v2` sentence transformer model.

Each hashtag is converted into a vector representation. These embeddings allow semantically related responses to be compared using cosine similarity, even when they do not share the exact same words.

In [37]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# Encode hashtags into vector embeddings using sentence transformer
long_df["embedding"] = list(
    model.encode(
        long_df["hashtag"].tolist(),
        convert_to_numpy=True   #Encode hashtags into NumPy embeddings
    )
)


/Users/nicoleluechin/anaconda3/envs/hashtag_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [12]:
long_df[["participant_id", "hashtag_slot", "hashtag", "scenario", "embedding"]].head()

,participant_id,hashtag_slot,hashtag,scenario,embedding
0,5c56964991c0ad0001cfa5ae,hashtag_1,safety,0,"[0.00057354796, 0.07829996, -0.057217155, -0.0..."
1,6980e62c2e406af9a4bd1fb0,hashtag_1,genetics,0,"[-0.03428222, 0.04356844, -0.052452214, 0.0873..."
2,6543d2b9d5cef1c67a0aa176,hashtag_1,crazytalk,0,"[-0.00644761, -0.030833134, 0.011760357, -0.02..."
3,5dd2f94a238a913096c52317,hashtag_1,flu,0,"[-0.018272426, 0.025064262, 0.0023866957, 0.05..."
4,5fa4f94e535f3b1aa628e0ab,hashtag_1,influenza,0,"[-0.046492066, 0.024335867, -0.039698552, 0.06..."


### 3. Semantic Similarity Analysis

This section examines whether participant responses become more semantically aligned under certain scenarios and whether that alignment changes across rounds.

In [13]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

#### 3a. Similarity by Scenario

For each scenario, average pairwise cosine similarity is computed across participant response embeddings.

Higher values suggest that participants interpreted the scenario in more similar semantic terms, while lower values suggest more diverse interpretations.

In [15]:
def avg_similarity_for_scenario(df, scenario_id):
    subset = df[df["scenario"] == scenario_id].copy()

    if len(subset) < 2:
        return None

    embeddings = np.vstack(subset["embedding"].values)
    sim_matrix = cosine_similarity(embeddings)

    mask = ~np.eye(sim_matrix.shape[0], dtype=bool)
    return sim_matrix[mask].mean()

In [17]:
for s in sorted(long_df["scenario"].unique()):
    print(f"Scenario {s}: {avg_similarity_for_scenario(long_df, s)}")

Scenario 0: 0.2307814210653305
Scenario 1: 0.2587577700614929
Scenario 2: 0.2244909405708313
Scenario 3: 0.2537640631198883
Scenario 4: 0.2288234829902649
Scenario 5: 0.23064908385276794
Scenario 6: 0.24001656472682953
Scenario 7: 0.22225530445575714


Similarity scores around ~0.22–0.26 indicate that responses are weakly to moderately related in meaning. This suggests that participants are generally responding within the same semantic space, but are not converging on highly similar or identical interpretations.

#### 3b. Similarity by Round

This analysis tests whether responses become more aligned or more dispersed over time.

If similarity increases across rounds, that would suggest convergence. If it decreases, that would suggest increasing semantic divergence.

In [20]:
def avg_similarity_by_round(df):
    results = {}

    for r in sorted(df["round"].unique()):
        subset = df[df["round"] == r]

        if len(subset) < 2:
            continue

        embeddings = np.vstack(subset["embedding"].values)
        sim_matrix = cosine_similarity(embeddings)

        mask = ~np.eye(sim_matrix.shape[0], dtype=bool)
        results[r] = sim_matrix[mask].mean()

    return results

long_df["round"] = long_df["hashtag_slot"].str.extract(r"hashtag_(\d+)")
long_df["round"] = long_df["round"].astype(int)

In [21]:
avg_similarity_by_round(long_df)

{1: 0.20383464,
 2: 0.19799507,
 3: 0.19789004,
 4: 0.1990662,
 5: 0.19897296,
 6: 0.19722074,
 7: 0.19679593,
 8: 0.19674142,
 9: 0.19533366,
 10: 0.1952367}

There’s a slight downward trend, but it’s pretty small, so I interpret it more as stability than meaningful divergence.

---
## How this connects to the chatbot system

The similarity analysis helps evaluate whether participant responses form a structured semantic space that can support the chatbot’s response mechanism. The chatbot maps user hashtags to a fixed set of semantic anchors using embedding similarity, which assumes that responses occupy a meaningful semantic space where related responses are closer together.  

The results show that similarity scores across scenarios fall within a narrow range (~0.22–0.26), indicating only modest differences in how consistently participants responded to different prompts, while similarity across rounds shows a slight decrease (~0.204 → ~0.195) but remains relatively stable overall.  

This suggests that responses are somewhat related in meaning but not tightly clustered, and that multiple interpretations are common. As a result, the chatbot must operate in a semantic space that is moderately structured but not strongly convergent: embedding-based similarity can still capture meaningful relationships, but mapping responses to a single anchor is not always unambiguous.  

This supports using a flexible, similarity-based approach for response selection rather than strict classification, while also highlighting the limitations of relying on a fixed set of semantic anchors in a setting where responses remain moderately diverse over time.

---

### 4. Interpretable Theme Mapping

To make the semantic pipeline interpretable, responses are compared against a small set of predefined theme anchors.

Each anchor represents a broader semantic category, such as:
- patient autonomy
- deception and trust
- fairness and rationing
- trauma and emotion
- medical ethics
- competition and status
- media and public reaction

Rather than allowing an unconstrained language model to generate responses freely, this approach maps user input onto a limited, interpretable theme space.

In [22]:
#Define anchors
theme_anchors = {
    "patient_autonomy": "patient rights, informed consent, respecting patient choice, patient decision making",
    "deception_trust": "lying to patients, withholding information, dishonesty, betrayal of trust",
    "fairness_rationing": "fairness in healthcare, justice, rationing treatment, scarce medical resources",
    "trauma_emotion": "pain, suffering, trauma, grief, emotional distress, severe injury",
    "medical_ethics": "bioethics, ethical dilemma in medicine, moral conflict in treatment decisions",
    "competition_status": "wealth, status, ambition, social climbing, paying to get ahead",
    "media_public_reaction": "media coverage, public controversy, public reaction, social opinion"
}

anchor_names = list(theme_anchors.keys())
anchor_texts = list(theme_anchors.values())
anchor_embeddings = model.encode(anchor_texts, convert_to_numpy=True)

In [23]:
#define response bank: what the bot is allowed to say back
response_bank = {
    "patient_autonomy": ["choice", "consent", "rights"],
    "deception_trust": ["truth", "honesty", "trust"],
    "fairness_rationing": ["justice", "equity", "fairness"],
    "trauma_emotion": ["pain", "grief", "trauma"],
    "medical_ethics": ["ethics", "dilemma", "morality"],
    "competition_status": ["status", "wealth", "ambition"],
    "media_public_reaction": ["controversy", "reaction", "opinion"]
}

### 5. Interpretable Bot Response Selection

The chatbot does not generate unconstrained text.

Instead, it:
1. cleans the user hashtag
2. embeds the input
3. compares it to the theme anchors
4. selects the highest-matching theme
5. returns a response from a restricted response bank tied to that theme

This makes the system easier to interpret, debug, and evaluate than an unconstrained generative LLM model.

In [24]:
#clean the input
def clean_hashtag_for_bot(tag):
    if tag is None:
        return None
    
    tag = str(tag).strip().lower().replace("#", "")
    tag = re.sub(r"[^a-z0-9_ ]+", "", tag)
    tag = re.sub(r"\s+", " ", tag).strip()
    
    return tag if tag != "" else None

In [27]:
#core response function
import random
def choose_bot_response(user_hashtag, model=model):
    cleaned = clean_hashtag_for_bot(user_hashtag)
    
    if cleaned is None:
        return {
            "input": user_hashtag,
            "cleaned_input": None,
            "predicted_label": None,
            "response": "response",
            "score_table": None
        }
    
    user_vec = model.encode([cleaned], convert_to_numpy=True)
    sims = cosine_similarity(user_vec, anchor_embeddings)[0]
    
    best_idx = sims.argmax()
    predicted_label = anchor_names[best_idx]
    
    score_table = pd.DataFrame({
        "label": anchor_names,
        "cosine_similarity": sims
    }).sort_values("cosine_similarity", ascending=False)
    
    response = random.choice(response_bank[predicted_label])
    
    return {
        "input": user_hashtag,
        "cleaned_input": cleaned,
        "predicted_label": predicted_label,
        "response": response,
        "score_table": score_table
    }

### Example Outputs

In [28]:
choose_bot_response("patientrights")

{'input': 'patientrights',
 'cleaned_input': 'patientrights',
 'predicted_label': 'patient_autonomy',
 'response': 'consent',
 'score_table':                    label  cosine_similarity
 0       patient_autonomy           0.626989
 1        deception_trust           0.442630
 2     fairness_rationing           0.348704
 4         medical_ethics           0.345242
 3         trauma_emotion           0.238614
 6  media_public_reaction           0.098242
 5     competition_status           0.078510}

In [30]:
choose_bot_response("truth")

{'input': 'truth',
 'cleaned_input': 'truth',
 'predicted_label': 'deception_trust',
 'response': 'truth',
 'score_table':                    label  cosine_similarity
 1        deception_trust           0.275179
 5     competition_status           0.215864
 6  media_public_reaction           0.213643
 0       patient_autonomy           0.212217
 3         trauma_emotion           0.174952
 4         medical_ethics           0.169376
 2     fairness_rationing           0.155036}

In [32]:
choose_bot_response("grief")

{'input': 'grief',
 'cleaned_input': 'grief',
 'predicted_label': 'trauma_emotion',
 'response': 'pain',
 'score_table':                    label  cosine_similarity
 3         trauma_emotion           0.498402
 1        deception_trust           0.287804
 6  media_public_reaction           0.169813
 4         medical_ethics           0.121346
 0       patient_autonomy           0.097796
 5     competition_status           0.090713
 2     fairness_rationing           0.000345}

In [33]:
choose_bot_response("wealth")

{'input': 'wealth',
 'cleaned_input': 'wealth',
 'predicted_label': 'competition_status',
 'response': 'status',
 'score_table':                    label  cosine_similarity
 5     competition_status           0.606061
 2     fairness_rationing           0.219650
 3         trauma_emotion           0.150452
 6  media_public_reaction           0.131528
 1        deception_trust           0.105831
 4         medical_ethics           0.104342
 0       patient_autonomy           0.082526}

---

## Supervised Extension: Random Forest Classification

So far, the chatbot system maps hashtags to semantic themes using embedding similarity and predefined anchors. 

To evaluate whether these theme assignments are learnable, this section extends the approach into a supervised setting. Each hashtag is assigned a theme label based on its closest anchor, and a Random Forest classifier is trained on the embeddings to predict those labels.

This allows us to test whether the semantic categories defined by the anchor system are structured enough to be learned by a standard machine learning model.

In [48]:
#Assign each hashtag a theme label based on closest semantic anchor (used as training labels)
long_df["anchor_label"] = long_df["hashtag"].apply(
    lambda x: choose_bot_response(x)["predicted_label"]
)

long_df[["hashtag", "anchor_label"]].head()

,hashtag,anchor_label
0,safety,trauma_emotion
1,genetics,medical_ethics
2,crazytalk,deception_trust
3,flu,trauma_emotion
4,influenza,media_public_reaction


In [42]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [43]:
# Expand embedding vectors into a feature matrix for model training
X = np.vstack(long_df["embedding"].values)
y = long_df["anchor_label"].values

print("Feature matrix shape:", X.shape)
print("Number of labels:", len(y))
pd.Series(y).value_counts()

Feature matrix shape: (15917, 384)
Number of labels: 15917


trauma_emotion           3950
competition_status       3440
deception_trust          2770
patient_autonomy         1814
media_public_reaction    1745
medical_ethics           1100
fairness_rationing       1098
Name: count, dtype: int64

In [44]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (12733, 384)
Test shape: (3184, 384)


In [45]:
# Train Random Forest classifier on hashtag embeddings
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [46]:
# Predict theme labels on the held-out test set
y_pred = rf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.8300879396984925

Classification Report:

                       precision    recall  f1-score   support

   competition_status       0.82      0.88      0.85       688
      deception_trust       0.86      0.71      0.78       554
   fairness_rationing       0.98      0.73      0.84       220
media_public_reaction       0.97      0.77      0.86       349
       medical_ethics       0.97      0.80      0.88       220
     patient_autonomy       0.93      0.78      0.85       363
       trauma_emotion       0.71      0.95      0.81       790

             accuracy                           0.83      3184
            macro avg       0.89      0.80      0.84      3184
         weighted avg       0.85      0.83      0.83      3184



### Interpretation

The Random Forest achieves approximately 83% accuracy, indicating that the anchor-based theme labels are largely learnable from the embedding space.

This suggests that the semantic structure captured by the embeddings is meaningful and consistent enough for a model to approximate the similarity-based mapping used by the chatbot.

In [47]:
# Show confusion matrix as a labeled table
cm = confusion_matrix(y_test, y_pred, labels=rf.classes_)
cm_df = pd.DataFrame(cm, index=rf.classes_, columns=rf.classes_)
cm_df

,competition_status,deception_trust,fairness_rationing,media_public_reaction,medical_ethics,patient_autonomy,trauma_emotion
competition_status,608,8,1,1,0,0,70
deception_trust,25,396,2,4,5,11,111
fairness_rationing,25,9,161,0,0,4,21
media_public_reaction,34,8,0,267,0,0,40
medical_ethics,9,6,0,2,176,4,23
patient_autonomy,12,23,1,0,0,282,45
trauma_emotion,25,10,0,1,0,1,753


This serves as a validation step for the overall system, showing that the semantic categories are not arbitrary and can be learned from the data./